# Worked solutions

This notebook is identical to the student version except that every 🔵 `# TODO` has been filled in, **with commentary on why the answer is what it is** rather than just the code. Read the comments — the reasoning is the point, not the syntax.

Everything else, including the ✏️ YOUR TURN cells, is unchanged: those have no single right answer.


# Module G — Brain transcriptomics

## Which genes differ in the Alzheimer's brain?

### What you will be able to do by the end

1. work with a dataset that has 2000 measurements and only 31 samples, and say why that is dangerous
2. use **unsupervised** methods — PCA and clustering — to find structure with no labels at all
3. run a differential-expression analysis with multiple-testing correction, and read a volcano plot
4. watch a classifier reach 100% accuracy on noise, and understand exactly how
5. explain why a 'downregulated neuronal gene' in AD tissue might mean no gene was regulated at all

### The data

**Real human brain tissue.** This is **GEO GSE1297** (Blalock et al., PNAS 2004): microarray measurements from post-mortem **hippocampal CA1** tissue of 31 people, graded from Control through Incipient and Moderate to Severe Alzheimer's disease. Each sample carries its real MMSE score, real Braak stage, real neurofibrillary tangle count, real age, sex and post-mortem interval.

We kept the 2000 most variable genes. Nothing is simulated. These are 31 real donated brains.

### How to work through this notebook

Run the cells in order, top to bottom. The notebook is split into four sections:

| | Section | What happens |
|---|---|---|
| 1 | **Understand the data** | Meet every column and every person in the table |
| 2 | **Quality control** | Find the flaws before they fool you |
| 3 | **Build models** | Start from something trivial, then climb |
| 4 | **Read the results** | Turn numbers into a clinical judgement |

Look out for these markers:

- ✏️ **YOUR TURN** — change the value shown, re-run the cell, watch the figure change. Everyone does these.
- 🟢 run and read · 🔵 write a little code · ⚫ take home
- 🧠 a question to think about; the answer is hidden underneath, so try first

**In a hurry?** Run section 1, then go to 3.1 (PCA) and 3.3 (differential expression) — those two carry the module.

---

*Teaching material. Nothing here is a diagnostic tool, and no result in this notebook is clinical evidence.*


In [ ]:
# Run me first. This finds the project folder, loads the shared helpers,
# and prints exactly where this module's data came from.
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plots
from data import load_data, load_extra, provenance
from models import split_data, train_model, evaluate, compare_models, sweep_parameter, MODEL_CHOICES

pd.set_option('display.width', 160)
print(provenance('G'))


---
# 1 · Understand the data

**This module is different from every other one today.** There is no patient to classify. The output is a list of genes and a picture — a *hypothesis*, not a prediction.


### 1.1 Two tables

`load_data('G')` gives the expression matrix: **one row per brain, one column per gene**. `load_extra('G')['samples']` gives what we know about each donor.

| Sample column | Meaning |
|---|---|
| `group` | Control / Incipient / Moderate / Severe — clinical severity at death. |
| `mmse` | Last Mini-Mental State Examination before death. |
| `braak_stage` | 0–6, how far tau tangles had spread through the brain at autopsy. The neuropathological gold standard. |
| `nft_count` | Neurofibrillary tangle density in this tissue. |
| `post_mortem_interval_h` | **Hours between death and tissue freezing.** RNA degrades. Remember this. |

The expression values are log2 microarray intensities: roughly 4 means barely detectable, 14 means abundant.


In [ ]:
expression = load_data('G').set_index('sample_id')
samples = load_extra('G')['samples'].set_index('sample_id')
samples = samples.loc[expression.index]

print(f'Expression matrix: {expression.shape[0]} brains x {expression.shape[1]} genes')
print(f'That is {expression.shape[1] // expression.shape[0]} times more measurements than samples.\n')
display(samples)


### 1.2 What p ≫ n means

Statisticians write **p ≫ n**: many more variables (p = 2000 genes) than observations (n = 31 brains). It is the defining condition of molecular biology data, and it breaks intuitions built on ordinary datasets.

Here is the consequence, in one figure: with 2000 genes and 31 samples, you can *always* find genes that separate any two groups perfectly — even groups you made up at random.


In [ ]:
rng = np.random.default_rng(0)
fake_label = rng.permutation([0] * 15 + [1] * 16)   # a meaningless coin-flip label

real_label = (samples['group'] != 'Control').to_numpy().astype(int)
values = expression.to_numpy()

def best_separation(labels):
    group_a, group_b = values[labels == 0], values[labels == 1]
    spread = np.sqrt(group_a.var(axis=0) / len(group_a) + group_b.var(axis=0) / len(group_b)) + 1e-9
    return np.abs(group_a.mean(axis=0) - group_b.mean(axis=0)) / spread

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.hist(best_separation(real_label), bins=50, alpha=0.7, color='#2c6fbb', label='real AD vs control label')
ax.hist(best_separation(fake_label), bins=50, alpha=0.7, color='#c0392b', label='a random made-up label')
ax.set_xlabel('separation between the two groups (t-like statistic)')
ax.set_ylabel('number of genes')
ax.set_title('With 2000 genes, even a meaningless label finds "good" genes')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(f'Genes separating the REAL groups at t > 3:   {(best_separation(real_label) > 3).sum()}')
print(f'Genes separating a RANDOM label at t > 3:    {(best_separation(fake_label) > 3).sum()}')
print('\nIf those two numbers are close, your "discovery" is arithmetic, not biology.')


### 1.3 ✏️ Your turn — look at one gene

Some of these genes are famous in Alzheimer's research. Try them, and try a few you have never heard of.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change GENE and re-run. Genes worth trying if they are present:
#     'GFAP'  - astrocyte marker, goes UP as glia react to damage
#     'SNAP25', 'SYT1' - synaptic genes, go DOWN as synapses are lost
#     'APOE', 'CLU', 'MAPT', 'APP' - the classic AD genes
#     'XIST'  - expressed from the inactive X. Try it and see what it separates!
#   The available genes are printed underneath if yours is missing.
# ==========================================================================
GENE = 'GFAP'

if GENE not in expression.columns:
    print(f'{GENE} is not among the 2000 most variable genes. A few that are:')
    print(', '.join(expression.columns[:40]))
else:
    frame = samples.copy()
    frame[GENE] = expression[GENE].to_numpy()
    plots.plot_by_group(frame, GENE, 'group', unit='(log2 intensity)',
                        title=f'{GENE} expression by clinical severity')
    plt.show()
    plots.plot_scatter(frame['mmse'], frame[GENE], colour_by=frame['group'],
                       xlabel='MMSE before death (30 = normal)', ylabel=f'{GENE} (log2)',
                       title=f'{GENE} against how impaired the person was', legend_title='group')
    plt.show()
    correlation = np.corrcoef(frame['mmse'], frame[GENE])[0, 1]
    print(f'Correlation with MMSE: {correlation:+.3f}   (n = 31 brains, so this is a noisy estimate)')


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** try `'XIST'` in 1.3 and work out what it is actually separating. (Hint: check the `sex` column.)
- 🔵 **If you want to write code:** rank all 2000 genes by their correlation with MMSE and print the top 20. Do any of them appear in the differential-expression results later?
- ⚫ **Take home:** GSE1297 measures whole tissue. Look up how single-nucleus RNA sequencing changed the picture — and what it costs.


---
# 2 · Quality control

Post-mortem tissue has failure modes that living-patient data does not.


### 2.1 The technical confounds

**Post-mortem interval** is how long the brain sat before freezing. RNA degrades in that window, unevenly across genes. If AD brains happened to have longer intervals — perhaps because those deaths occurred in nursing homes rather than hospitals — then 'degraded RNA' would masquerade as 'AD biology'.

**Age** is the other one. AD donors are older. Age changes brain expression on its own.


In [ ]:
samples['is_ad'] = (samples['group'] != 'Control').astype(int)
for column, unit in [('post_mortem_interval_h', '(hours)'), ('age', '(years)'), ('braak_stage', '(0-6)')]:
    plots.plot_by_group(samples.assign(status=np.where(samples.is_ad == 1, 'AD', 'control')),
                        column, 'status', unit=unit,
                        title=f'{column} in AD versus control donors')
    plt.show()

print(samples.groupby('is_ad')[['age', 'post_mortem_interval_h', 'braak_stage', 'mmse']].mean().round(2))
print('\nIf these differ between the groups, they are confounded with diagnosis.')


### 2.2 The biological confound — you are counting cells, not measuring regulation

This is the deepest point in the module and it is easy to miss.

A tissue sample is a **mixture** of neurons, astrocytes, microglia and oligodendrocytes. Bulk expression measures the average over that mixture. In an Alzheimer's hippocampus **there are fewer neurons** — they have died — and **more reactive glia**.

So if a neuronal gene looks 'downregulated in AD', there are two completely different explanations:

1. Each surviving neuron is expressing less of it — a *regulatory* change.
2. Each surviving neuron expresses exactly as much as before, but there are fewer neurons in the tube — a *compositional* change.

**Bulk data cannot distinguish these.** Many published 'AD gene signatures' are substantially measuring cell loss. Below we make the problem visible using marker genes.


In [ ]:
markers = {
    'neurons': ['SNAP25', 'SYT1', 'RBFOX3', 'NEFL', 'SYN1', 'STMN2'],
    'astrocytes': ['GFAP', 'AQP4', 'S100B', 'SLC1A2', 'ALDH1L1'],
    'microglia': ['AIF1', 'CD68', 'ITGAM', 'CSF1R', 'C1QB'],
}
available = {kind: [gene for gene in genes if gene in expression.columns]
             for kind, genes in markers.items()}
for kind, genes in available.items():
    print(f'{kind}: found {len(genes)} marker gene(s) — {", ".join(genes) if genes else "none in the top 2000"}')

scores = pd.DataFrame({kind: expression[genes].mean(axis=1)
                       for kind, genes in available.items() if genes})
scores['status'] = np.where(samples['is_ad'].to_numpy() == 1, 'AD', 'control')

for kind in [column for column in scores.columns if column != 'status']:
    plots.plot_by_group(scores, kind, 'status', unit='(mean log2 of marker genes)',
                        title=f'Average {kind} marker expression — a proxy for how many {kind} are in the tube')
    plt.show()


🧠 **Think first:** The neuronal markers are lower in AD tissue. Have neuronal genes been switched off?

<details>
<summary>Click for one good answer</summary>

You cannot tell from this data, and that is the honest answer. Fewer neurons in the sample produces exactly this figure, with no change in gene regulation at all. To separate the two you need either single-cell/single-nucleus sequencing (measure each cell separately) or computational **deconvolution** (estimate the mixture and adjust for it). Any bulk-tissue paper that claims a regulatory mechanism without addressing composition is making a claim its data cannot support.

</details>


### 2.3 QC verdict

**Usable for hypothesis generation only.** Three caveats that go in every sentence we write about this data:

1. n = 31. Every estimate is noisy, and the multiple-testing burden is severe.
2. Post-mortem interval and age differ between groups and cannot be fully adjusted at this sample size.
3. Cell-composition change is indistinguishable from regulatory change.

And a fourth that is not a flaw but a limit: **this is end-stage tissue.** These brains are from people who died with advanced disease. Whatever we find came *after* decades of pathology, so nothing here can tell us about cause.

*(**Express path:** you can start from section 3 — run its catch-up cell first and everything below stands alone.)*


---
# 3 · Unsupervised discovery

In every other module today we told the algorithm the answer and asked it to learn the rule. Here we tell it **nothing** and ask whether it finds structure by itself. If the brains separate by diagnosis without ever being told the diagnosis, that is real evidence of a molecular difference.


### 🚏 Taking the Express path? Run this one cell first

It rebuilds everything sections 3 and 4 need, so you can start here without having run sections 1 and 2 yourself. **If you did run them, run this anyway** — it just redefines the same things and costs a second.


In [ ]:
# Express catch-up: safe to run whether or not you did sections 1 and 2.
expression = load_data('G').set_index('sample_id')
samples = load_extra('G')['samples'].set_index('sample_id').loc[expression.index]
samples['is_ad'] = (samples['group'] != 'Control').astype(int)
print(f'{expression.shape[0]} brains x {expression.shape[1]} genes, '
      f'{int(samples.is_ad.sum())} of them from donors with Alzheimer disease.')
print('Ready for section 3.')


### 3.1 PCA — 2000 dimensions squeezed into 2

**Principal component analysis** finds the directions along which the samples differ most, and lets us plot 2000-dimensional data on a page. PC1 is the single biggest axis of variation between these brains — whatever it happens to be.

**Predict before you run:** will PC1 line up with diagnosis? With age? With post-mortem interval? Nothing tells the algorithm which one to find.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(expression.to_numpy())
pca = PCA(n_components=6, random_state=42).fit(scaled)
coordinates = pca.transform(scaled)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(range(1, 7), 100 * pca.explained_variance_ratio_, color='#2c6fbb')
ax.set_xlabel('principal component'); ax.set_ylabel('percent of variation explained')
ax.set_title('How much of the difference between brains each axis captures')
plt.tight_layout(); plt.show()

for colour_column, label in [('group', 'clinical severity'), ('sex', 'sex'),
                             ('post_mortem_interval_h', 'post-mortem interval'), ('braak_stage', 'Braak stage')]:
    values = samples[colour_column]
    if values.dtype.kind in 'if':
        fig, ax = plt.subplots(figsize=(6.2, 4.4))
        scatter = ax.scatter(coordinates[:, 0], coordinates[:, 1], c=values, cmap='viridis', s=70,
                             edgecolor='white')
        fig.colorbar(scatter, ax=ax, label=label)
        ax.set_xlabel(f'PC1 ({100 * pca.explained_variance_ratio_[0]:.0f}% of variation)')
        ax.set_ylabel(f'PC2 ({100 * pca.explained_variance_ratio_[1]:.0f}%)')
        ax.set_title(f'31 brains, coloured by {label}')
        plt.tight_layout()
    else:
        plots.plot_scatter(coordinates[:, 0], coordinates[:, 1], colour_by=values,
                           xlabel=f'PC1 ({100 * pca.explained_variance_ratio_[0]:.0f}% of variation)',
                           ylabel=f'PC2 ({100 * pca.explained_variance_ratio_[1]:.0f}%)',
                           title=f'31 brains, coloured by {label}', legend_title=label)
    plt.show()


**Whatever you see here is the finding.** If the groups separate along PC1, the molecular difference is the dominant source of variation between these brains. If they do not — if PC1 turns out to track sex, or post-mortem interval, or nothing recognisable — that is a *more* important result, because it tells you that any 'AD signature' extracted from this tissue is a minority of the variation and is competing with technical noise.


### 3.2 ✏️ Your turn — clustering without labels

**k-means** splits the samples into `k` groups purely by similarity. It has never seen a diagnosis. If its groups line up with the clinical ones, that is genuine unsupervised discovery.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change these and re-run.
#     N_CLUSTERS: try 2, 3, 4. There are 4 clinical groups - does k=4 recover them?
#     N_GENES:    try 100, 500, 2000. Does using more genes help, or add noise?
#   Look at the crosstab: perfect agreement would be one number per row.
# ==========================================================================
N_CLUSTERS = 2
N_GENES = 2000

from sklearn.cluster import KMeans

most_variable = expression.var().sort_values(ascending=False).index[:N_GENES]
subset = StandardScaler().fit_transform(expression[most_variable].to_numpy())
clusters = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit_predict(subset)

reduced = PCA(n_components=2, random_state=42).fit_transform(subset)
plots.plot_scatter(reduced[:, 0], reduced[:, 1], colour_by=[f'cluster {c}' for c in clusters],
                   xlabel='PC1', ylabel='PC2',
                   title=f'k-means found {N_CLUSTERS} groups using {N_GENES} genes — no labels involved',
                   legend_title='unsupervised cluster')
plt.show()

agreement = pd.crosstab(samples['group'], clusters)
print('Rows = the real clinical group. Columns = what the algorithm decided.')
display(agreement)

purity = agreement.max(axis=1).sum() / agreement.to_numpy().sum()
print(f'Cluster purity: {purity:.2f}  (1.0 would mean the clusters exactly match the clinical groups)')


### 3.3 Differential expression — which genes, specifically?

Now a gene-by-gene test: for each of the 2000 genes, is its average different between AD and control brains? That is 2000 t-tests, so we **must** correct for multiple testing — at p < 0.05 we would expect 100 false positives from noise alone.

We use the **Benjamini–Hochberg false discovery rate**: instead of controlling the chance of *any* false positive, it controls the *proportion* of your findings that are false. An FDR of 0.05 means "about 5% of the genes on this list are wrong", which is the right trade-off for hypothesis generation.


In [ ]:
from scipy import stats

ad_values = expression[samples['is_ad'] == 1].to_numpy()
control_values = expression[samples['is_ad'] == 0].to_numpy()

statistic, raw_p = stats.ttest_ind(ad_values, control_values, axis=0, equal_var=False)
log_fold_change = ad_values.mean(axis=0) - control_values.mean(axis=0)   # already log2, so a difference IS a ratio

# Benjamini-Hochberg, written out rather than imported, so you can see it.
order = np.argsort(raw_p)
ranks = np.arange(1, len(raw_p) + 1)
adjusted = np.minimum.accumulate((raw_p[order] * len(raw_p) / ranks)[::-1])[::-1]
fdr = np.empty_like(adjusted)
fdr[order] = np.clip(adjusted, 0, 1)

results = pd.DataFrame({'gene': expression.columns, 'log2_fold_change': log_fold_change,
                        'p_value': raw_p, 'fdr': fdr}).sort_values('p_value')

print(f'Genes with raw p < 0.05:      {(raw_p < 0.05).sum():4d}   <- of which ~{int(0.05 * len(raw_p))} are noise')
print(f'Genes with FDR < 0.05:        {(fdr < 0.05).sum():4d}   <- the defensible list')
print(f'Genes with FDR < 0.10:        {(fdr < 0.10).sum():4d}\n')

plots.plot_volcano(results['log2_fold_change'], results['fdr'], results['gene'].tolist(),
                   alpha=0.05, top=14,
                   title='Volcano plot: right = higher in AD, up = more certain')
plt.show()
display(results.head(15).round(4))


🧠 **Think first:** Why not just report the genes with raw p < 0.05, and mention the caveat in the discussion?

<details>
<summary>Click for one good answer</summary>

Because with 2000 tests you expect about 100 of them to pass at p < 0.05 **when nothing is going on at all** — section 1.2 showed exactly that with a made-up label. A list of a hundred genes, of which a hundred could be noise, is not a finding with a caveat; it is a caveat with no finding.

Benjamini–Hochberg changes what you are promising. Rather than *"probably no false positives"* (Bonferroni, which at n = 31 would leave you with nothing), it promises *"about 5% of this list is wrong"* — which is honest, achievable, and exactly the right guarantee when the output is a shortlist of hypotheses that somebody will now test at the bench.

</details>


### 3.4 ✏️ Your turn — the heatmap

The classic transcriptomics figure. Each row is a gene, each column a brain, colour is expression relative to that gene's average.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change these and re-run.
#     N_TOP:    how many genes to show (try 15, 30, 60)
#     ORDER_BY: 'braak_stage', 'mmse', 'group' or 'post_mortem_interval_h'
#   If the colour pattern lines up with the ordering, the genes track it.
# ==========================================================================
N_TOP = 30
ORDER_BY = 'braak_stage'

top_genes = results.head(N_TOP)['gene'].tolist()
column_order = samples.sort_values(ORDER_BY).index
block = expression.loc[column_order, top_genes]
z_scores = ((block - block.mean()) / block.std()).transpose()

column_labels = [f"{sample.split('GSM')[-1]} {samples.loc[sample, 'group'][:4]} "
                 f"{ORDER_BY[:4]}={samples.loc[sample, ORDER_BY]}" for sample in column_order]
plots.plot_heatmap(z_scores.to_numpy(), top_genes, column_labels,
                   title=f'Top {N_TOP} differentially expressed genes, brains ordered by {ORDER_BY}')
plt.show()


### 3.5 The cautionary demonstration — a classifier that cannot fail

Suppose you ignored everything in section 2 and trained a classifier on all 2000 genes. With 31 samples it will fit perfectly. Here is the point: **it fits perfectly on made-up labels too.**


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

pipeline = Pipeline([('scale', StandardScaler()), ('svm', SVC(kernel='linear', C=1.0))])
X_genes = expression.to_numpy()
real_y = samples['is_ad'].to_numpy()
fake_y = np.random.default_rng(1).permutation(real_y)   # same labels, shuffled: pure noise

pipeline.fit(X_genes, real_y)
training_accuracy = pipeline.score(X_genes, real_y)
pipeline.fit(X_genes, fake_y)
fake_training_accuracy = pipeline.score(X_genes, fake_y)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
real_cv = cross_val_score(pipeline, X_genes, real_y, cv=folds, scoring='balanced_accuracy').mean()
fake_cv = cross_val_score(pipeline, X_genes, fake_y, cv=folds, scoring='balanced_accuracy').mean()

plots.plot_score_comparison(
    ['real labels\ntraining score', 'SHUFFLED labels\ntraining score',
     'real labels\ncross-validated', 'SHUFFLED labels\ncross-validated'],
    [training_accuracy, fake_training_accuracy, real_cv, fake_cv],
    colours=['#c0392b', '#c0392b', '#2c6fbb', '#2c6fbb'], reference=0.5,
    title='2000 genes, 31 brains. The red bars are meaningless; only the blue ones are evidence.',
    ylabel='balanced accuracy')
plt.show()
print('Both red bars are 1.00. A model that perfectly classifies random noise has')
print('demonstrated nothing except that p >> n lets you draw a line anywhere.')


### 3.6 🔵 Your turn to write code — is the signal above chance?

The honest way to test an unsupervised finding: a **permutation test**. Shuffle the labels many times, redo the analysis, and ask how often noise produces a result as good as yours.

Fill in the `# TODO`.


In [ ]:
# ✅ Worked solution.
def count_significant(labels, alpha=0.05):
    a = expression[labels == 1].to_numpy()
    b = expression[labels == 0].to_numpy()
    _, p = stats.ttest_ind(a, b, axis=0, equal_var=False)
    order = np.argsort(p)
    ranks = np.arange(1, len(p) + 1)
    adjusted = np.minimum.accumulate((p[order] * len(p) / ranks)[::-1])[::-1]
    return int((np.clip(adjusted, 0, 1) < alpha).sum())

real_count = count_significant(samples['is_ad'].to_numpy())
generator = np.random.default_rng(7)
labels = samples['is_ad'].to_numpy()
null_counts = [count_significant(generator.permutation(labels)) for _ in range(30)]

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(null_counts, bins=15, color='#cccccc', label='shuffled labels (noise)')
ax.axvline(real_count, color='#c0392b', linewidth=2, label=f'real labels ({real_count} genes)')
ax.set_xlabel('genes passing FDR < 0.05'); ax.set_ylabel('how often')
ax.set_title('Permutation test: is the real finding outside the noise distribution?')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()
print(f'Empirical p-value: {(np.sum(np.array(null_counts) >= real_count) + 1) / (len(null_counts) + 1):.3f}')

# Why this is the right test. The FDR correction already assumes a particular null model
# (independent tests). Gene expression is emphatically NOT independent - genes in a pathway
# rise and fall together - so the analytic FDR can be optimistic. A permutation test makes
# no such assumption: it destroys the label-to-sample link while preserving every correlation
# between genes, which is exactly the null hypothesis we care about.
#
# Read the histogram, not just the p-value. If the shuffled runs regularly return dozens of
# 'significant' genes, then a list of dozens is not a discovery, whatever the FDR column says.


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 3.4, set `ORDER_BY = 'post_mortem_interval_h'` and check whether the pattern also lines up with a purely technical variable.
- 🔵 **If you want to write code:** repeat the differential expression using **Braak stage** as a continuous variable (correlate each gene with Braak) rather than a binary AD/control split. Neuropathology is graded, not binary.
- ⚫ **Take home:** take your top gene list to a free enrichment tool (Enrichr, g:Profiler, DAVID) and see which pathways come out. Then ask whether those pathways are neuronal, glial, or immune — and what that implies given section 2.2.


---
# 4 · Read the results

A discovery module's deliverable is **a gene list, a figure, and a carefully hedged sentence**. There is no accuracy score, and inventing one would misrepresent what was done.


### 4.1 Your findings


In [ ]:
significant = results[results['fdr'] < 0.05]
up = significant[significant['log2_fold_change'] > 0].head(10)
down = significant[significant['log2_fold_change'] < 0].head(10)

print(f'{len(significant)} genes differ at FDR < 0.05 between {int(samples.is_ad.sum())} AD '
      f'and {int((1 - samples.is_ad).sum())} control hippocampi.\n')

combined = pd.concat([up, down])
plots.plot_importance(combined['gene'], combined['log2_fold_change'],
                      title='Top differentially expressed genes (blue = higher in AD)',
                      xlabel='log2 fold change, AD versus control')
plt.show()

print('Higher in AD:', ', '.join(up['gene'].tolist()) or 'none')
print('Lower in AD: ', ', '.join(down['gene'].tolist()) or 'none')


### 4.2 ✏️ Your turn — does your gene track severity, or just diagnosis?

A gene that scales smoothly with Braak stage or MMSE is a much stronger candidate than one that merely differs between two crude groups.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Pick any gene from your top list above.
#   Then change AGAINST to 'braak_stage', 'mmse', 'nft_count'
#   or 'post_mortem_interval_h'.
#   A convincing candidate tracks the pathology AND NOT the
#   post-mortem interval. Check both before you believe it.
# ==========================================================================
GENE_OF_INTEREST = results.iloc[0]['gene']
AGAINST = 'braak_stage'

frame = samples.copy()
frame['expression'] = expression[GENE_OF_INTEREST].to_numpy()
plots.plot_scatter(frame[AGAINST], frame['expression'], colour_by=frame['group'],
                   xlabel=AGAINST, ylabel=f'{GENE_OF_INTEREST} (log2 intensity)',
                   title=f'{GENE_OF_INTEREST} against {AGAINST}', legend_title='clinical group')
plt.show()

usable = frame[[AGAINST, 'expression']].dropna()
r, p = stats.pearsonr(usable[AGAINST], usable['expression'])
print(f'{GENE_OF_INTEREST} vs {AGAINST}: r = {r:+.3f}, p = {p:.4f}  (n = {len(usable)} brains)')
print('With 31 samples, treat any single correlation as a hint, not a result.')


### 4.3 What this module produced, and what it did not

**What you can honestly say:**

- A list of genes whose average expression differs between AD and control hippocampal tissue at a controlled false discovery rate.
- A picture showing how much — or how little — of the total variation between these brains is explained by diagnosis.
- A demonstration that a classifier on this data can reach perfect training accuracy on random labels.

**What you cannot say:**

- That any of these genes *causes* Alzheimer's disease. This is end-stage post-mortem tissue; everything observed came decades after the disease began.
- That a gene was *regulated* up or down. It may simply reflect which cells survived (2.2).
- That the finding will replicate. n = 31, and independent replication in transcriptomics is hard.

**The honest framing** is the one this module exists to teach: *these genes are worth looking at next.* That is what discovery science produces. It is upstream of everything else in today's menu — the biomarkers in module C and the drug targets in module H exist because analyses like this pointed at them first.

**Ethics.** These are 31 donated human brains. Consent for brain donation is given by the donor before death or by family after it, and covers research use — but donors could not have anticipated every future analysis. Cohorts like this also skew towards people connected to academic medical centres, which is its own kind of unrepresentativeness.

---

### 🧠 Final question for the group discussion

Modules A–F all try to **predict** something about a patient. This one tries to **understand** something about a disease. **Which one is more useful?** And: if a drug company had to choose between funding a better diagnostic model and funding a study like this one, which should they pick?


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** in 4.2, check your top gene against `post_mortem_interval_h` as well as `braak_stage`. Does it survive?
- 🔵 **If you want to write code:** adjust the differential expression for post-mortem interval by regressing it out of each gene first, then redo the t-tests. How many genes survive?
- ⚫ **Take home:** compare your gene list with a published AD single-nucleus atlas and see which of your genes turn out to be markers of a cell type rather than a disease process.
